In [1]:
#import relevant libraries
import os
#from scipy import stats

import numpy as np
#import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB
import NLMATH
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap


import dabest
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

#NOTE: SUPPRESSES WARNINGS!

import warnings


warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)


Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 13.20it/s]


Numba compilation complete!


In [2]:
#initial file processing
labcomp = "C:\\Users\\User"
computer2 = "C:\\Users\\lnico"
officecomp = "C:\\Users\\Star"
homecomp = "D:"
titledpath = homecomp


filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\"
savedir = titledpath + filedir + "Compilation with delta\\2025deltagcollection\\"
saveosardir = titledpath + filedir + "Compilation with delta\\2025fallingtoosarcomp\\"
    
openPath = titledpath + filedir
files = os.listdir(openPath)

#identifying genotypes
responder = "ACR"
respondercsv = responder + ".csv"
wt = "w1118"


In [6]:
lstnew=[]

#lstnew should be the list of names you want to process the files with. Only choose one

#if you want to process all the names in the filedir

for file_no in os.listdir(openPath): 
    if respondercsv in file_no and "w1118" not in file_no :   
        f = os.path.join(openPath, file_no)
        dfe=pd.read_csv(f)
        exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
        driver = file_no.split(" ")[0]
        lstnew.append(driver)
#lst = lstnew.copy()
lst = [x for x in lstnew if x not in ['Th-Gal4', 'R58']]

#processing ONLY specific names
# lst = ["MB112C"]
lst = ['MB399B']
print(lst)

['MB399B']


In [7]:
for n in lst:
    driver = n
    print(n)
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    #adjust this depending on timeframe
    dfexpt = NLCLIMB.timerule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.timerule(NLCLIMB.generation(wtdf, wt))
       
    #processing before dabest application 
    df_f = NLMATH.fallingocc(dfexpt, dfwt).reset_index(drop=True)
    df_sp = NLMATH.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_bsp = NLMATH.bspeed(NLMATH.boutspeed(dfexpt), NLMATH.boutspeed(dfwt)).reset_index(drop=True)
    df_h = NLMATH.totalheight(dfexpt, dfwt).reset_index(drop=True)
    df_pp = NLMATH.bheight(NLMATH.pauseheight(dfexpt), NLMATH.pauseheight(dfwt)).reset_index(drop=True)
    df_maxv = pd.concat([NLMATH.maxvelocity(dfexpt, "Expt"), NLMATH.maxvelocity(dfwt, "WT")], axis = 0).reset_index(drop=False)
    df_sim = pd.concat([NLMATH.straightnessindexmeter(dfexpt, "Expt"), NLMATH.straightnessindexmeter(dfwt, "WT")], axis = 0).reset_index(drop=True)

    #pause and bouts
    wttotalmeanevent, wttotalnumberevent = NLMATH.pausecomp(dfwt, wt)
    expttotalmeanevent, expttotalnumberevent = NLMATH.pausecomp(dfexpt, driver)
    alltgtmeandf_bout = pd.concat([NLMATH.pausenumber(wttotalmeanevent, n, "Bouts"), NLMATH.pausenumber(expttotalmeanevent, n, "Bouts")], axis = 0).reset_index(drop=True)
    alltgtnumberdf_bout = pd.concat([NLMATH.pausenumber(wttotalnumberevent, n, "Bouts"), NLMATH.pausenumber(expttotalnumberevent, n, "Bouts")], axis = 0).reset_index(drop=True)
            
    #___________________________________________#    
    # meandiff plots -- you run mean_diff instead of delta_g because since all the binary data is at the same dimension, no standardization is required and empirical delta delta is sufficient
    #dff2_prop = NLMATH.deltaversion_meandiff(df_f, "binary_fallvalue", "fallprop")
    dff2_number = NLMATH.deltaversion_meandiff(df_f, "Fall", "fallnumber")   #number of falls is under deltaversion_binary because the number of flies that fall could actually be so few in number, that the SD is 0, and thus hedges g will not be able to perform since the divisor ==0

    #deltag plots
    dfs2 = NLMATH.deltaversion_deltag(df_sp, "Velocity", "speed")
    dfh2 = NLMATH.deltaversion_deltag(df_h, "Y", "height")
    dfbs2 = NLMATH.deltaversion_deltag(df_bsp, "BSpeed", "bspeed")
    dfpp2 = NLMATH.deltaversion_deltag(df_pp, "Height", "pausepos")
    dfmv2 = NLMATH.deltaversion_deltag(df_maxv, "maxvelocity", "maxvelocity")
    dfsim2 = NLMATH.deltaversion_deltag(df_sim, "averagestraightnessindex", "straightindex")
    #pause and bouts
    dfmb2 = NLMATH.deltaversion_deltag(alltgtmeandf_bout, "Bouts", "meanbout")     
    dfnb2 = NLMATH.deltaversion_deltag(alltgtnumberdf_bout, "Bouts", "bout")
    
    #singledelta processing
    lsr_bsp = NLMATH.log2metric(df_bsp, "BSpeed")
    lsr_sp = NLMATH.log2metric(df_sp, 'Velocity')    
    
    #new index and ratio metrics
    bout_index_nb = NLMATH.boutindex(alltgtnumberdf_bout, "Bouts") 
    ratio_mb = NLMATH.simplemetricratio(alltgtmeandf_bout, "Bouts") 
    ratio_maxv = NLMATH.simplemetricratio(df_maxv, "maxvelocity")
    

    df_lsrbsp = NLMATH.singledelta(lsr_bsp, "log2 BSpeed", "log2bspeed")
    df_lsrsp = NLMATH.singledelta(lsr_sp, "log2 Velocity", "log2speed")
    df_boutindex_nb = NLMATH.singledelta(bout_index_nb, "Bouts", "boutnumber_index")  # Renamed from ratio to index
    df_ratio_mb = NLMATH.singledelta(ratio_mb, "Bouts", "boutduration_ratio")
    df_ratio_maxv = NLMATH.singledelta(ratio_maxv, "maxvelocity", "maxvelocity_ratio")
    
    #final df and saving into excel
    dftotal = pd.concat([dff2_number, dfs2, dfh2, dfbs2, dfpp2, dfmv2, dfsim2, dfmb2, dfnb2, df_lsrbsp, df_lsrsp, df_boutindex_nb, df_ratio_mb, df_ratio_maxv], axis = 1)
    dftotal['MBON'] = n
    dftotal.set_index("MBON", inplace = True)
    #dftotal.to_csv(savedir + n + " x " + responder + "_deltag_allstats.csv")
    
print("Done!")        

MB399B
Done!


In [8]:
dftotal

,fallnumber_bootstrap,fallnumber_meandiff,speed_bootstrap,speed_deltag,height_bootstrap,height_deltag,bspeed_bootstrap,bspeed_deltag,pausepos_bootstrap,pausepos_deltag,...,log2bspeed_bootstrap,log2bspeed_hedgesg,log2speed_bootstrap,log2speed_hedgesg,boutnumber_index_bootstrap,boutnumber_index_hedgesg,boutduration_ratio_bootstrap,boutduration_ratio_hedgesg,maxvelocity_ratio_bootstrap,maxvelocity_ratio_hedgesg
MBON,,,,,,,,,,,,,,,,,,,,,
MB399B,-0.333333,-0.422,-0.158831,-0.264,0.453397,0.316,0.053219,-0.046,0.427229,0.283,...,-0.136133,-0.006,0.004559,0.185,0.068257,-0.041,0.021366,0.217,0.622201,0.613
MB399B,-0.473193,-0.422,-0.420663,-0.264,0.083989,0.316,-0.142749,-0.046,0.258323,0.283,...,-0.085677,-0.006,-0.026933,0.185,-0.019351,-0.041,0.062333,0.217,0.461095,0.613
MB399B,-0.580420,-0.422,-0.225763,-0.264,0.399204,0.316,0.030257,-0.046,0.397983,0.283,...,0.230481,-0.006,0.152400,0.185,-0.045102,-0.041,-0.046391,0.217,0.639613,0.613
MB399B,-0.463869,-0.422,-0.399123,-0.264,0.086860,0.316,-0.228919,-0.046,0.074069,0.283,...,-0.276699,-0.006,0.317492,0.185,0.254928,-0.041,0.175180,0.217,0.755487,0.613
MB399B,-0.794872,-0.422,-0.248827,-0.264,0.516707,0.316,-0.234046,-0.046,0.218431,0.283,...,-0.110581,-0.006,0.031800,0.185,-0.233485,-0.041,0.392629,0.217,0.727658,0.613
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MB399B,-0.235431,-0.422,-0.419919,-0.264,0.061657,0.316,-0.068448,-0.046,0.074210,0.283,...,0.106457,-0.006,0.175625,0.185,0.072864,-0.041,-0.043555,0.217,0.704149,0.613
MB399B,0.214452,-0.422,-0.157368,-0.264,0.515514,0.316,-0.193054,-0.046,0.476934,0.283,...,-0.193950,-0.006,0.320393,0.185,0.200487,-0.041,0.225526,0.217,0.661711,0.613
MB399B,-0.468531,-0.422,-0.378647,-0.264,0.014705,0.316,0.092974,-0.046,0.220431,0.283,...,0.152648,-0.006,0.072383,0.185,-0.146792,-0.041,0.301238,0.217,0.508512,0.613


In [ ]:
def thesisexcel(df_sp, metric):
    df6 = df_sp[(df_sp['ExperimentState'] != "Recovery") ]
    name = []
    if any(df6[metric].isnull()):
        name = df6[df6[metric].isnull()]['index'].tolist()
    dfsp_db = df6[~df6['index'].isin(name)]
            
    #dfsp_db2 = dabest.load(data = dfsp_db, x = ['ExperimentState', 'ExperimentState'], paired = "baseline", id_col="index", y = metric, delta2 = True, experiment = "Type", x1_level = ["Dark", "Full"], experiment_label = ["WT","Expt"] )
    dfsp_db2 = dabest.load(data = dfsp_db, x = ["ExperimentState", "Type"], y = metric,  delta2 = True, experiment = "Type",
                            experiment_label = ['WT', 'Expt'], x1_level = ["Dark", "Full"], paired = "baseline", id_col="index" ) #if delta2 = dabest; deltaG = dabest_jck
    
    if 'WT' in dfsp_db2.hedges_g.results.control[0]:
        wt_idx = 0
        expt_idx = 1
    else:
        wt_idx = 1
        expt_idx = 0

    dfthesis_wt = pd.DataFrame({"MBON": [n], 
                                    "Genotype": "w1118;;UAS-" + responder + "/+\nw1118;" + n + "/+;" + n + "/+", 
                                    "Sample Size": dfsp_db2.hedges_g.results.control_N[wt_idx],
                                    "Hedge's g":f"{dfsp_db2.hedges_g.results.difference[wt_idx]:.2f}\n[{dfsp_db2.hedges_g.results.bca_low[wt_idx]:.2f}, {dfsp_db2.hedges_g.results.bca_high[wt_idx]:.2f}]",
                                    "Delta g":[" "]})

    dfthesis_expt = pd.DataFrame({"MBON": [n], 
                                "Genotype": ["w1118;" + n + "/+;" + n + "/" + responder], 
                                "Sample Size": [dfsp_db2.hedges_g.results.control_N[expt_idx]],
                                "Hedge's g": [f"{dfsp_db2.hedges_g.results.difference[expt_idx]:.2f}\n[{dfsp_db2.hedges_g.results.bca_low[expt_idx]:.2f}, {dfsp_db2.hedges_g.results.bca_high[expt_idx]:.2f}]"],
                                "Delta g": [f"{dfsp_db2.hedges_g.delta_delta.results.difference[0]:.2f}\n[{dfsp_db2.hedges_g.delta_delta.results.bca_low[0]:.2f}, {dfsp_db2.hedges_g.delta_delta.results.bca_high[0]:.2f}]"]
                                })

    dfthesis = pd.concat([dfthesis_wt, dfthesis_expt], ignore_index=True)

    return (dfthesis)

-0.5369648701108664 -0.16136652564941661


In [53]:
if 'WT' in dfsp_db2.hedges_g.results.control[0]:
    wt_idx = 0
    expt_idx = 1
else:
    wt_idx = 1
    expt_idx = 0

dfthesis_wt = pd.DataFrame({"MBON": [n], 
                                "Genotype": "w1118;;UAS-" + responder + "/+\nw1118;" + n + "/+;" + n + "/+", 
                                "Sample Size": dfsp_db2.hedges_g.results.control_N[wt_idx],
                                "Hedge's g":f"{dfsp_db2.hedges_g.results.difference[wt_idx]:.2f}\n[{dfsp_db2.hedges_g.results.bca_low[wt_idx]:.2f}, {dfsp_db2.hedges_g.results.bca_high[wt_idx]:.2f}]",
                                "Delta g":[" "]})

dfthesis_expt = pd.DataFrame({"MBON": [n], 
                              "Genotype": ["w1118;" + n + "/+;" + n + "/" + responder], 
                              "Sample Size": [dfsp_db2.hedges_g.results.control_N[expt_idx]],
                              "Hedge's g": [f"{dfsp_db2.hedges_g.results.difference[expt_idx]:.2f}\n[{dfsp_db2.hedges_g.results.bca_low[expt_idx]:.2f}, {dfsp_db2.hedges_g.results.bca_high[expt_idx]:.2f}]"],
                              "Delta g": [f"{dfsp_db2.hedges_g.delta_delta.results.difference[0]:.2f}\n[{dfsp_db2.hedges_g.delta_delta.results.bca_low[0]:.2f}, {dfsp_db2.hedges_g.delta_delta.results.bca_high[0]:.2f}]"]
                              })

dfthesis = pd.concat([dfthesis_wt, dfthesis_expt], ignore_index=True)

In [54]:
dfthesis

,MBON,Genotype,Sample Size,Hedge's g,Delta g
0,MB112C,w1118;;UAS-ACR/+\nw1118;MB112C/+;MB112C/+,210,"0.77\n[0.63, 0.91]",
1,MB112C,w1118;MB112C/+;MB112C/ACR,97,"0.50\n[0.32, 0.67]","-0.35\n[-0.54, -0.16]"


In [50]:
dfthesis_wt

,MBON,Genotype,Sample Size,Hedge's g,Delta g
0,MB112C,w1118;;UAS-ACR/+\nw1118;MB112C/+;MB112C/+,210,"0.77\n[0.63, 0.91]",


In [52]:
dfthesis_expt

,MBON,Genotype,Sample Size,Hedge's g,Delta g
0,MB112C,w1118;MB112C/+;MB112C/ACR,97,"0.50\n[0.32, 0.67]","-0.35\n[-0.54, -0.16]"


In [30]:
dfsp_db2.hedges_g.results.difference[0].values()

AttributeError: 'numpy.float64' object has no attribute 'values'

In [20]:
dfsp_db2.hedges_g.results.control

0      Dark WT
1    Dark Expt
Name: control, dtype: object